# 03 — Preprocessing (and data leakage)

## Data leakage

1. **What it is:** using information at training time that would not be available — or would be computed using the test distribution — in a fair evaluation.
2. **Why we care:** leaked models look brilliant in a notebook and fail quietly in production (or on a true holdout).
3. **How it happens here:** scaling on the full dataset; SMOTE before splitting; dropping rows using test labels; using `Class` as a feature.
4. **In this project:** `StandardScaler` lives inside a `Pipeline` and is **fit on training folds only**. Resampling happens **after** the split, **inside** training.

```
Raw rows
   ↓
Drop exact duplicates
   ↓
Deterministic features (hour, log amount)  ← no learned statistics
   ↓
Stratified train / test split
   ↓
Fit scaler + model on TRAIN
   ↓
Transform test with the *train* scaler
```


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data_processing import dataset_overview, drop_duplicates_if_present, load_raw_dataset
from src.utils import RAW_DATASET_PATH

from src.data_processing import build_model_pipeline, stratified_train_test_split
from src.feature_engineering import MODEL_FEATURE_COLUMNS
from sklearn.linear_model import LogisticRegression

df = drop_duplicates_if_present(load_raw_dataset())
X_train, X_test, y_train, y_test = stratified_train_test_split(df)
print(X_train.shape, X_test.shape)
print("train fraud rate:", float(y_train.mean()))
print("test fraud rate:", float(y_test.mean()))
print("features:", MODEL_FEATURE_COLUMNS)


## Stratification

**What:** split so that class proportions in train and test match the original.

**Why ordinary random split is risky:** with ~500 fraud rows, an unlucky split can put almost all fraud in train or test. Metrics then swing wildly.

**How:** `train_test_split(..., stratify=y)`.

## Scaling

Linear models and k-NN (inside SMOTE) care about feature scale. Trees care less, but a **shared pipeline** keeps evaluation honest.

We still scale trees so SMOTE / logistic regression comparisons are fair.


In [ ]:
pipe = build_model_pipeline(LogisticRegression(max_iter=1000, class_weight="balanced"))
pipe


## Missing values and duplicates

This file has no missing cells (verify in notebook 01). Duplicates are dropped **before** splitting so the same fingerprint cannot appear on both sides.

If you ever impute, fit the imputer on **train only**, same as the scaler.


In [ ]:
print("NaNs in train:", int(X_train.isna().sum().sum()))
print("NaNs in test:", int(X_test.isna().sum().sum()))
